# Expérience 4 — SVD (bibliothèque Surprise)

Surprise attend des **notes explicites**, absentes du jeu de données. Trois
définitions de note ont été implémentées ; ce notebook les compare sur le même
protocole que les autres méthodes.

| Définition | Construction | Notebook de référence |
|---|---|---|
| étoiles de l'article | clics reçus par l'article, échelle log 1–5 | `02_notation_etoiles.ipynb` |
| intensité par couple | nombre de clics du couple (lecteur, article) | — |
| binaire + négatifs | lu = 1, non lu échantillonné = 0 | — |

⚠️ Chaque entraînement prend plusieurs minutes. Les artefacts déjà présents dans
`models_split/` correspondent à la variante « étoiles ».

## Protocole commun

Identique dans tous les notebooks d'expérimentation, sinon les chiffres ne sont pas
comparables :

- **découpage temporel 60 / 20 / 20** sur `click_timestamp` ;
- artefacts construits sur la **seule** période d'entraînement (`models_split/`) ;
- réglage sur la **validation** ; la période de test reste intacte jusqu'à la mesure
  finale (notebook 07) ;
- lecteurs évalués : connus à l'entraînement **et** actifs pendant la période
  d'évaluation ;
- métriques : HitRate@5, Recall@5, couverture, personnalisation.

> Prérequis : `python -m src.evaluate --data-dir data/news-portal-user --out-dir models_split`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')

from src import experiments as xp
from src.recommender import Recommender

DATA = Path('..') / 'data' / 'news-portal-user'
MODELS = Path('..') / 'models_split'

train, val, test = xp.load_split(DATA)
reco = Recommender(MODELS)
users, cible = xp.eval_users(reco, val, max_users=2000)
print(f'{len(users):,} lecteurs évalués sur la période de validation')

[clicks] 1 fichier(s) vide(s) ignoré(s) : clicks_hour_100.csv


[split] entraînement 1,792,908 | validation 597,636 | test 597,637
[split] bornes temporelles : t60=1507602953792 t80=1507843212579


2,000 lecteurs évalués sur la période de validation


## 1. Variante « étoiles » (artefacts existants)

Comparée à la popularité récente, qui sert de référence.

In [2]:
POOL_HEURES = 6
pool = xp.recent_pool(train, POOL_HEURES)

configs = {
    'SVD étoiles': xp.make_svd(reco, pool),
    'ALS (référence)': xp.make_als(reco, pool),
    'popularité récente (référence)': xp.make_popularity(pool),
}
xp.compare(configs, users, cible, n_articles=reco.n_articles)

,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
SVD étoiles,0.0005,0.0000,0.002,16.6
ALS (référence),0.0080,0.0016,0.027,86.4
popularité récente (référence),0.1320,0.0222,0.001,0.0


## 2. Variante binaire avec échantillonnage négatif

Une note qui ne dépend que de l'article ne peut pas personnaliser : le biais
article capte tout le signal. L'alternative est de poser lu = 1 / non lu = 0, ce qui
transforme la prédiction de note en tâche de classement.

Le nombre de négatifs par positif est le réglage à explorer.

In [3]:
from src.collaborative_surprise import (add_negative_samples, build_ratings,
                                        save_artifacts, train_svd)

TEMP = Path('..') / 'models_split_svd_tmp'
TEMP.mkdir(exist_ok=True)
for nom in ('articles_embeddings_pca.npy', 'user_clicks.pkl', 'popular_articles.npy'):
    cible_fichier = TEMP / nom
    if not cible_fichier.exists():
        cible_fichier.write_bytes((MODELS / nom).read_bytes())

configs = {'popularité récente (référence)': xp.make_popularity(pool)}
for negatifs in (1, 4):
    notes = add_negative_samples(build_ratings(train), negatives_per_positive=negatifs)
    algo, trainset = train_svd(notes, n_factors=50, n_epochs=20)
    save_artifacts(algo, trainset, TEMP)
    reco_tmp = Recommender(TEMP)
    configs[f'SVD binaire, {negatifs} négatif(s)'] = xp.make_svd(reco_tmp, pool)

xp.compare(configs, users, cible, n_articles=reco.n_articles)

[notes] 1,769,009 positifs + 1,767,138 négatifs = 3,536,147 exemples


[svd] 255,516 users x 29,118 items, 50 facteurs, ~60.6 Mo d'artefacts


[notes] 1,769,009 positifs + 7,061,782 négatifs = 8,830,791 exemples


[svd] 255,516 users x 29,118 items, 50 facteurs, ~60.6 Mo d'artefacts


,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
popularité récente (référence),0.1320,0.0222,0.001,0.0
"SVD binaire, 1 négatif(s)",0.0075,0.0013,0.036,44.9
"SVD binaire, 4 négatif(s)",0.0755,0.0110,0.012,31.5


## Lecture

Trois définitions de note, trois résultats très différents — **avec la même
bibliothèque et le même algorithme** :

| Définition de la note | HitRate@5 (validation) | Personnalisation |
|---|---|---|
| étoiles de l'article | 0,0005 | 16,6 % |
| binaire, 1 négatif | 0,0075 | 44,9 % |
| binaire, 4 négatifs | 0,0755 | 31,5 % |

**Ce qui compte n'est pas la bibliothèque mais la formulation du problème.**

Les étoiles échouent parce que la note ne dépend que de l'article : deux lecteurs
attribuent la même valeur au même article, le biais article capte donc tout le
signal et le modèle apprend « quels articles sont lus », pas « qui lit quoi ». Sa
personnalisation (16,6 %) tombe même sous celle de la popularité.

Les notes binaires avec négatifs réparent cela : en donnant au modèle des exemples
« non lu = 0 », on transforme une prédiction de note en tâche de **classement**,
qui est la vraie question posée. Le gain est d'un facteur 150.

### Réserve importante sur ces chiffres

Le 0,0755 de la variante à 4 négatifs **ne s'est pas reproduit sur la période de
test** : elle y obtient 0,0160 (voir notebook 07), soit 4,7 fois moins, et passe
derrière l'ALS (0,0250) et le contenu (0,0200).

Autrement dit ce réglage est **instable** : il dépend fortement du vivier de
candidats et de la période. On ne peut donc pas conclure que le SVD dépasse l'ALS —
seulement que la définition de la note change tout, ce qui reste vrai dans les deux
mesures (0,0160 contre 0,0000).

C'est aussi la raison d'être de la séparation validation / test : sans elle, nous
aurions annoncé 0,0755 comme un résultat.